# Transcribing audio with Whisper on a Colab T4

Pull an audio file from a Drive link and stand up a Hugging Face `transformers` Whisper
pipeline ready to transcribe it. None of the Whisper checkpoints are gated, so there is no
Hugging Face token to set up.

**Pick a GPU runtime first:** *Runtime -> Change runtime type -> T4 GPU*. `ffmpeg`, which
Whisper uses to decode almost any audio or video container, is already installed on Colab.

In [ ]:
# Colab preinstalls transformers, but the pinned version drifts; large-v3-turbo needs >=4.44.
# gdown is already present on Colab and is what pulls the file from a Drive link.
!pip install -q -U transformers accelerate python-dotenv

In [ ]:
# Imports

import gc
import os
from functools import lru_cache
from getpass import getpass
from pathlib import Path
from threading import Thread

import gdown
import torch
from dotenv import load_dotenv
from IPython.display import Markdown, display
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TextIteratorStreamer,
    pipeline,
)

try:
    from google.colab import userdata
except ImportError:
    userdata = None  # not on Colab — the .env / prompt fallbacks cover it

In [ ]:
# Constants

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
# Login to HF


def hf_login() -> None:
    """Log in to Hugging Face — needed for gated models like Llama 3.2.

    Looks for the token in three places, in order: a local .env file (handy when running
    off Colab), Colab's secret store (only reachable from the Colab UI in a browser tab —
    driving the runtime from an IDE makes userdata.get raise), and finally a getpass prompt.
    Nothing typed at the prompt is saved into the notebook.
    """
    load_dotenv()
    token = os.getenv("HF_TOKEN")

    if not token and userdata is not None:
        try:
            token = userdata.get("HF_TOKEN")
        except Exception as exc:
            print(f"Colab secret unavailable ({type(exc).__name__}).")

    if not token:
        token = getpass("HF token: ").strip()

    login(token)
    print("Logged in to Hugging Face.")


hf_login()

In [ ]:
def as_pipeline_input(file_id: str, output: str = "audio_input") -> str:
    """Download the shared Drive file and return it in the form the ASR pipeline expects.

    file_id is the long token in the share URL: drive.google.com/file/d/<THIS_PART>/view.
    gdown is used rather than a plain request because Drive serves an HTML interstitial to
    naive clients; the output name's extension does not matter — ffmpeg sniffs the format.

    The pipeline type-checks its input with isinstance(..., str), so a Path object slips
    through and errors; this resolves the download, confirms it landed, and returns a str.
    """
    gdown.download(id=file_id, output=output, quiet=False)
    resolved = Path(output).expanduser().resolve()
    if not resolved.is_file():
        raise FileNotFoundError(
            f"No audio file at {resolved} — did the download succeed?"
        )
    return str(resolved)


file_id = "1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU"
audio_path = as_pipeline_input(file_id)

In [ ]:
# Transcribe audio


@lru_cache(maxsize=1)
def get_transcriber(model: str = "openai/whisper-large-v3-turbo"):
    """Build a Whisper ASR pipeline once and reuse it across calls.

    float16 on a CUDA T4 (a Turing card with no bf16), float32 on CPU. No chunk_length_s
    here on purpose: that switches on the fast-but-experimental chunked algorithm, which can
    misread words at the 30s chunk seams. Leaving it off lets the pipeline hand the whole
    file to model.generate, which runs Whisper's own sequential long-form algorithm —
    sliding a 30s window and carrying context across it, as in the original paper.
    """
    if torch.cuda.is_available():
        device, dtype = "cuda:0", torch.float16
    else:
        device, dtype = "cpu", torch.float32

    return pipeline(
        "automatic-speech-recognition",
        model=model,
        dtype=dtype,
        device=device,
    )


def transcribe(audio_path: str, language: str = "en") -> str:
    """Transcribe an audio file with the Whisper pipeline and return the plain text.

    return_timestamps=True is what triggers the sequential long-form path for audio over 30
    seconds; without it the pipeline would only transcribe the first 30. language pins
    Whisper instead of letting it auto-detect from the opening seconds; pass None to
    auto-detect, or e.g. "it" / "es" for other languages.
    """
    result = get_transcriber()(
        audio_path,
        return_timestamps=True,
        generate_kwargs={"language": language, "task": "transcribe"},
    )
    return result["text"].strip()


transcript = transcribe(audio_path)
print(transcript)

In [ ]:
# Meeting minutes with Llama


def meeting_minutes(
    transcript: str, model_id: str = LLAMA, max_new_tokens: int = 800
) -> str:
    """Turn a transcript into markdown meeting minutes with Llama 3.2, streaming as it goes.

    Llama 3.2 3B loads in fp16 (~6.5 GB), which sits comfortably on a 16 GB T4 next to the
    cached Whisper pipeline, so no quantization is needed. TextIteratorStreamer hands the
    generated text back as a Python iterator: generate() blocks until it is done, so it runs
    on a background thread while the main thread reads the streamer — printing each chunk live
    and accumulating the full string to return. The GPU is freed at the end.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You write concise, well-structured meeting minutes from a transcript. "
                "Respond in markdown with a short summary, a list of key discussion points, "
                "takeaways, and action items with owners and deadlines where they are stated."
            ),
        },
        {"role": "user", "content": f"Here is the transcript:\n\n{transcript}"},
    ]

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")

    llm = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=torch.float16, device_map="auto"
    )

    # skip_prompt so the stream is the reply only, not the templated prompt echoed back;
    # skip_special_tokens drops the chat markers Llama emits around its turn.
    streamer = TextIteratorStreamer(
        tokenizer, skip_prompt=True, skip_special_tokens=True
    )
    thread = Thread(
        target=llm.generate,
        kwargs=dict(**inputs, max_new_tokens=max_new_tokens, streamer=streamer),
    )
    thread.start()

    minutes = ""
    try:
        for chunk in streamer:
            print(chunk, end="")
            minutes += chunk
    finally:
        thread.join()
        del llm
        gc.collect()
        torch.cuda.empty_cache()
    return minutes


minutes = meeting_minutes(transcript)

In [ ]:
# Render the minutes as formatted markdown rather than the raw streamed text
display(Markdown(minutes))